## Launching model on dataset and collecting output

### Imports

In [ ]:
from datasets import load_dataset
from tqdm import tqdm

from langchain_core.prompts import ChatPromptTemplate
from services.common.llm_interface import LLMInterface
from transformers import AutoModelForCausalLM, AutoTokenizer
from huggingface_hub import login
import os
import torch
import sys

sys.path.append("../../../services/")
from index import Index
from services.experiment.cropped.data_process_utils import retrieve_answer_token_index

torch.random.manual_seed(42)

/workspace/Diplom/.cursor_venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
os.environ.setdefault('CUDA_VISIBLE_DEVICES', '3')
print(f"CUDA_VISIBLE_DEVICES={os.environ['CUDA_VISIBLE_DEVICES']}")

### Preparation

In [ ]:
ITERATIONS_COUNT = 12000

In [ ]:
torch.random.manual_seed(42)

device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
if torch.cuda.is_available():
    torch.cuda.set_device(device)
print(device)
print(torch.cuda.get_device_properties(device))

CUDA_VISIBLE_DEVICES=0
cuda:0
_CudaDeviceProperties(name='NVIDIA L40', major=8, minor=9, total_memory=45385MB, multi_processor_count=142, uuid=441e6008-cd27-34b5-c81a-17dad9d4a894, L2_cache_size=96MB)


In [4]:
dataset = load_dataset("TIGER-Lab/MMLU-Pro", split="test")

'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/TIGER-Lab/MMLU-Pro/resolve/main/README.md (Caused by ProxyError(\'Unable to connect to proxy\', NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fb6d955f830>: Failed to resolve \'host.docker.internal\' ([Errno -2] Name or service not known)")))'), '(Request ID: 9a652374-70aa-4d29-b183-ad009197ed4d)')' thrown while requesting HEAD https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro/resolve/main/README.md
Retrying in 1s [Retry 1/5].


'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/TIGER-Lab/MMLU-Pro/resolve/main/README.md (Caused by ProxyError(\'Unable to connect to proxy\', NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fb46923b5c0>: Failed to resolve \'host.docker.internal\' ([Errno -2] Name or service not known)")))'), '(Request ID: 59f68a0e-1f19-405f-93da-325c32ed769d)')' thrown while requesting HEAD https://huggingface.co/datasets/TIGER-Lab/MMLU-Pro/resolve/main/README.md
Retrying in 2s [Retry 2/5].
'(MaxRetryError('HTTPSConnectionPool(host=\'huggingface.co\', port=443): Max retries exceeded with url: /datasets/TIGER-Lab/MMLU-Pro/resolve/main/README.md (Caused by ProxyError(\'Unable to connect to proxy\', NameResolutionError("<urllib3.connection.HTTPSConnection object at 0x7fb46923b440>: Failed to resolve \'host.docker.internal\' ([Errno -2] Name or service not known)")))'), '(Request ID: 1c0c020b-ef85-4809-ad59-d482175cb5

KeyboardInterrupt: 

In [ ]:
model_id = "Qwen/Qwen2.5-7B-Instruct"
model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    attn_implementation="eager",
    torch_dtype=torch.bfloat16
)
tokenizer = AutoTokenizer.from_pretrained(model_id)
model = model.to(device)
model.eval()

Sliding Window Attention is enabled but not implemented for `eager`; unexpected results may be encountered.
Loading checkpoint shards: 100%|██████████| 4/4 [00:00<00:00, 75.74it/s]


Qwen2ForCausalLM(
  (model): Qwen2Model(
    (embed_tokens): Embedding(152064, 3584)
    (layers): ModuleList(
      (0-27): 28 x Qwen2DecoderLayer(
        (self_attn): Qwen2Attention(
          (q_proj): Linear(in_features=3584, out_features=3584, bias=True)
          (k_proj): Linear(in_features=3584, out_features=512, bias=True)
          (v_proj): Linear(in_features=3584, out_features=512, bias=True)
          (o_proj): Linear(in_features=3584, out_features=3584, bias=False)
        )
        (mlp): Qwen2MLP(
          (gate_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (up_proj): Linear(in_features=3584, out_features=18944, bias=False)
          (down_proj): Linear(in_features=18944, out_features=3584, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
        (post_attention_layernorm): Qwen2RMSNorm((3584,), eps=1e-06)
      )
    )
    (norm): Qwen2RMSNorm((3584,), eps=1e-06)
    (rotary_emb):

In [ ]:
chat_llm = LLMInterface(
    model=model,
    tokenizer=tokenizer,
    device=device
)

prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert at answering multiple choice questions. 
            You will be given a question and options. Follow these rules strictly:
            
            1. Select the correct option number between 0 and 9
            2. Return your response as SINGLE token, only ONE number between 0 and 9

            Follow this output format:
            OPTION_NUMBER
            
            Example:
            2 

            Do not include any additional text except answer number between 0 and 9
            Answer the following question:
            """,
        ),
        (
            "human",
            """
            Question: {input_question}
    
            Options:
            {input_options}
            """,
        ),
    ]
)


llm_chain = prompt | chat_llm

In [ ]:
len(dataset)

12032

### Running model on dataset and collecting results


In [ ]:
def filter_func(model_response, right_answer):
    answer_token_index = retrieve_answer_token_index(model_response["score_data"])
    if answer_token_index is None or answer_token_index >= len(model_response["score_data"]) - 1:
        return False

    token_data = model_response["score_data"][answer_token_index]
    return right_answer in token_data["top_tokens"]

In [ ]:
# launching model

ERROR_LIMIT = 500

index = Index(f"../../../index_data/qwen2.5_MMLU-PRO_attn_cropped_{ITERATIONS_COUNT}")
index.clear()

correct_count = 0
progress_bar = tqdm(
    enumerate(dataset.select(range(ITERATIONS_COUNT))),
    total=ITERATIONS_COUNT,
    desc="Scoring MMLU-PRO",
)

for iter, data in progress_bar:
    for error_count in range(ERROR_LIMIT + 1):
        try:
            response = llm_chain.invoke(
                {
                    "input_question": dataset[iter]["question"],
                    "input_options": [
                        f"{i}:  {x}"
                        for i, x in enumerate(dataset[iter]["options"])
                    ]
                }
            )
            
            response["dataset_elem"] = dataset[iter]
        
            if filter_func(response, str(data["answer_index"])):
                if response["output_text"][-1] == str(
                    data["answer_index"]
                ):
                    correct_count += 1

                progress_bar.set_postfix(
                    {"Current accuracy": str(correct_count / (iter + 1))}
                )
                
                response = {
                    "iteration": iter,
                    **response
                }

                index.save_data(response, iter, logging=False)
                break
            
            if error_count == ERROR_LIMIT:
                raise Exception("Error limit exceeded")
            
        except Exception as e:
            if (error_count + 1) % 500 == 0:
                print(f"Error on iteration {iter + 1}\nError count: {error_count}\nMessage: {e}")
            if error_count == ERROR_LIMIT:
                print("Error limit exceeded")

File is deleted: ../../../index_data/qwen2.5_MMLU-PRO_attn_cropped_12000_index.pkl
File is deleted: ../../../index_data/qwen2.5_MMLU-PRO_attn_cropped_12000_data.pkl
Index is cleared successfully.


Scoring MMLU-PRO:   0%|          | 0/12000 [00:00<?, ?it/s]

Scoring MMLU-PRO: 100%|██████████| 12000/12000 [32:46<00:00,  6.10it/s, Current accuracy=0.34775]            


In [ ]:
print(len(index))

12000
